# Case 6: Business Context Defteri

Case 1-5, tamamen **etiketsiz** bir disiplinle ilerledi: `isFraud` hiçbir tasarım kararını yönlendirmedi. Case 6 farklı bir katman ekliyor: Case 5'in ürettiği `final_raw_anomaly_score`'u **iş bağlamına** göre yeniden ağırlıklandırıp false positive oranını azaltmak. İstatistiksel anomaliyi iş açısından anlamlı anomaliye çeviren bir politika katmanı, dört istatistiksel katmanın üzerine değil yanına ekleniyor.

Bu ilk madde: business hours (iş saatleri) bağlamı.

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "requirements.txt").exists():
            return parent
    raise RuntimeError("repo root not found: expected a requirements.txt somewhere above " + str(start))


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

PosixPath('/home/canberk/workspace/case_study')

In [2]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from src.config import settings

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

parquet_path = settings.processed_data_path / "merged_transactions.parquet"
print(f"kaynak: {parquet_path}")

kaynak: /home/canberk/workspace/case_study/data/processed/merged_transactions.parquet


## 1. Business Hours Context

**Business hours tanımı (sabit, standart bir iş günü kuralı; veriden türetilmemiş):** hafta içi + 09:00-18:00.

**Tasarım gerilimi:** "mesai dışını nasıl düzeltelim" sorusuna iki prensip zıt cevap veriyor:

1. **Hacim/güven tabanlı (etiketsiz):** Mesai dışı saatlerde işlem hacmi çok düşük (Case 1: saatlik hacim \~15 kata varan fark); segment bazlı istatistikler az gözleme dayanıyor, Case 4'te üç kez bağımsız keşfedilen "az örnek = gürültülü skor" dersiyle aynı mantık. Bu prensip mesai dışı skorları **hafifletmeyi** önerir.
2. **Fraud-oranı tabanlı (`isFraud` kullanıyor; bilinçli ve açık bir istisna):** Bu veri setinde mesai dışı saatlerde fraud oranı fiilen **daha yüksek** (Case 1'in bulgusu). Düzeltme yönü bunun tam tersini, mesai dışı skorları **artırmayı** önerir.

İkisi zıt yönde çalıştığı için tek birini seçip diğerini elemek yerine, **ikisini de ayrı fonksiyon olarak uyguluyoruz**; Case 4/5'teki "iki yöntemi karşılaştır" deseninin devamı.

In [3]:
from src.services.anomaly.combined import compute_all_anomaly_scores, PRIMARY_SCORE_COLUMNS
from src.services.anomaly.normalization import normalize_scores
from src.services.anomaly.aggregation import compute_final_raw_anomaly_score
from src.services.features.temporal import build_temporal_features

all_scores = compute_all_anomaly_scores(parquet_path)
normalized = normalize_scores(all_scores, PRIMARY_SCORE_COLUMNS)
final_raw = compute_final_raw_anomaly_score(normalized, PRIMARY_SCORE_COLUMNS)
temporal = build_temporal_features(parquet_path)
isfraud = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "isFraud"]).to_pandas()

print(f"final_raw şekil: {final_raw.shape}")

final_raw şekil: (590540, 2)


### Yöntem 1: Hacim/güven tabanlı (etiketsiz)

In [4]:
from src.services.anomaly.business_context import apply_confidence_based_adjustment

confidence_adjusted = apply_confidence_based_adjustment(final_raw, temporal)
print("iş saatlerinde çarpan (hepsi 1,0 olmalı):", confidence_adjusted[confidence_adjusted["is_business_hours"]]["confidence_multiplier"].unique())
print("mesai dışında çarpan aralığı:", confidence_adjusted[~confidence_adjusted["is_business_hours"]]["confidence_multiplier"].describe()[["min", "max"]].to_dict())
confidence_adjusted.head(10)

iş saatlerinde çarpan (hepsi 1,0 olmalı): [1.]
mesai dışında çarpan aralığı: {'min': 0.5294313190074795, 'max': 1.0}


,TransactionID,business_adjusted_score_confidence_based,confidence_multiplier,is_business_hours
0,2987000,0.153260,0.948712,False
1,2987001,0.326056,0.948712,False
2,2987002,0.215794,0.948712,False
3,2987003,0.407779,0.948712,False
4,2987004,0.215426,0.948712,False
5,2987005,0.225296,0.948712,False
6,2987006,0.396341,0.948712,False
7,2987007,0.647145,0.948712,False
8,2987008,0.521846,0.948712,False
9,2987009,0.345572,0.948712,False


İş saatlerinde çarpan tam olarak 1,0: skor değişmiyor. Mesai dışında 0,53-1,0 arası (üst sınır, üst-%1'lik ham skorların hafifletmeden muaf tutulmasından geliyor; aşağıda doğrulanacak).

### Yöntem 2: Fraud-oranı tabanlı (`isFraud` kullanıyor, açıkça işaretli)

In [5]:
from src.services.anomaly.business_context import apply_fraud_rate_calibrated_adjustment

fraud_calibrated = apply_fraud_rate_calibrated_adjustment(final_raw, temporal, isfraud)
boost = fraud_calibrated[~fraud_calibrated["is_business_hours"]]["fraud_calibration_multiplier"].unique()
print(f"mesai dışı artırma çarpanı (tek bir sabit, oran-tabanlı): {boost}")
fraud_calibrated.head(10)

mesai dışı artırma çarpanı (tek bir sabit, oran-tabanlı): [1.29995004]


,TransactionID,business_adjusted_score_fraud_calibrated,fraud_calibration_multiplier,is_business_hours
0,2987000,0.210001,1.29995,False
1,2987001,0.446771,1.29995,False
2,2987002,0.295687,1.29995,False
3,2987003,0.558749,1.29995,False
4,2987004,0.295182,1.29995,False
5,2987005,0.308706,1.29995,False
6,2987006,0.543077,1.29995,False
7,2987007,0.886735,1.29995,False
8,2987008,0.715047,1.29995,False
9,2987009,0.473512,1.29995,False


Mesai dışı işlemler \~%30 artırılıyor (1,30); bu veri setinde mesai-dışı/mesai-içi fraud oranı oranından direkt hesaplandı. **Bunun projenin geri kalanından farkı önemli:** buraya kadar hiçbir katman/adım `isFraud`'u bir parametreyi belirlemek için kullanmadı; bu fonksiyon bilinçli bir istisna. Gerçek bir üretim sisteminde böyle bir kalibrasyonun ayrı, etiketli bir doğrulama sürecinden (train/validation ayrımı, zaman içinde kayma kontrolü vb.) geçmesi gerekir; burada sadece karşılaştırma amaçlı, notebook'un kapsamı dışında.

### Doğrulama 1: üst-%1, hacim/güven yönteminde hiç değişmiyor mu?

In [6]:
check = final_raw.merge(confidence_adjusted[["TransactionID", "business_adjusted_score_confidence_based"]], on="TransactionID")
check = check.merge(fraud_calibrated[["TransactionID", "business_adjusted_score_fraud_calibrated"]], on="TransactionID")

raw_top1 = check["final_raw_anomaly_score"] >= check["final_raw_anomaly_score"].quantile(0.99)
conf_top1 = check["business_adjusted_score_confidence_based"] >= check["business_adjusted_score_confidence_based"].quantile(0.99)
print(f"raw top-%1 ile confidence-based top-%1 örtüşmesi: {(raw_top1 & conf_top1).sum()} / {raw_top1.sum()}")

raw top-%1 ile confidence-based top-%1 örtüşmesi: 5906 / 5906


**Tam eşleşme (5906/5906): beklenen, sürpriz değil.** Tasarım gereği üst-%1'lik ham skorlar hiç hafifletilmiyor, dolayısıyla o küme değişmeden kalıyor. Gerçek etkiyi görmek için daha geniş bir dilime (%5) bakmak gerekiyor.

### Doğrulama 2: üst-%5'te ne değişiyor, ve nereye gidiyor

In [7]:
raw_top5 = check["final_raw_anomaly_score"] >= check["final_raw_anomaly_score"].quantile(0.95)
conf_top5 = check["business_adjusted_score_confidence_based"] >= check["business_adjusted_score_confidence_based"].quantile(0.95)

left_set = check[raw_top5 & ~conf_top5].merge(temporal[["TransactionID", "hour_of_day"]], on="TransactionID")
entered_set = check[~raw_top5 & conf_top5].merge(temporal[["TransactionID", "hour_of_day"]], on="TransactionID")

print(f"raw top-%5: {raw_top5.sum()} işlem")
print(f"kümeden çıkan: {len(left_set)}, kümeye giren: {len(entered_set)}")
print(f"\nçıkanların saat dağılımı:\n{left_set['hour_of_day'].value_counts().sort_index().head(8)}")
print(f"\ngirenlerin saat dağılımı:\n{entered_set['hour_of_day'].value_counts().sort_index().tail(8)}")

raw top-%5: 29527 işlem
kümeden çıkan: 6042, kümeye giren: 6042

çıkanların saat dağılımı:
hour_of_day
0    356
1    936
2    933
3    832
4    583
5    427
6    261
7    185
Name: count, dtype: int64

girenlerin saat dağılımı:
hour_of_day
16    489
17    635
18    599
19    773
20    692
21    580
22    472
23    173
Name: count, dtype: int64


**Mekanizma tam beklendiği gibi çalışıyor:** kümeden çıkan \~6.042 işlem neredeyse tamamen saat 0-3 (düşük hacimli, mesai dışı); hafifletme onları üst-%5'in dışına itiyor. Kümeye giren \~6.042 işlem ise saat 17-19'da yoğunlaşıyor (mesai bitimine yakın, hâlâ yüksek hacimli); bunlar zaten yüksekti, göreli olarak öne çıktılar. Bu, "az veriye dayanan gürültülü yüksek skorları hafiflet" tasarım hedefinin somut kanıtı.

### Betimleyici kontrol: hangi yöntem fraud oranıyla daha iyi hizalanıyor?

In [8]:
check_fraud = check.merge(isfraud, on="TransactionID")

for col in ["final_raw_anomaly_score", "business_adjusted_score_confidence_based", "business_adjusted_score_fraud_calibrated"]:
    check_fraud[f"{col}_decile"] = pd.qcut(check_fraud[col], 10, labels=[str(i) for i in range(1, 11)])
    rates = (check_fraud.groupby(f"{col}_decile", observed=True)["isFraud"].mean() * 100).round(2)
    print(f"{col}:\n{rates.to_dict()}\n")

final_raw_anomaly_score:
{'1': 1.82, '2': 2.01, '3': 2.16, '4': 2.69, '5': 3.08, '6': 3.51, '7': 4.29, '8': 4.51, '9': 5.57, '10': 5.34}



business_adjusted_score_confidence_based:
{'1': 2.0, '2': 2.35, '3': 2.62, '4': 3.07, '5': 3.37, '6': 3.68, '7': 3.96, '8': 4.33, '9': 4.8, '10': 4.81}



business_adjusted_score_fraud_calibrated:
{'1': 1.86, '2': 1.92, '3': 2.18, '4': 2.61, '5': 3.06, '6': 3.36, '7': 4.1, '8': 4.62, '9': 5.38, '10': 5.89}



**Üçünü karşılaştırmalı okuyalım:**

- `final_raw_anomaly_score`: %1,82 → %5,34 (9-10. kovada hafif dalgalanma).
- `business_adjusted_score_confidence_based`: %2,00 → %4,81, düşük kovalarda biraz daha yüksek başlıyor, üst kovalarda **düzleşiyor** (4,80/4,81: neredeyse fark yok). Beklenen: hafifletme, mesai-dışı yanlış-pozitifleri orta kovalara doğru taşıyor, en üst kovanın saflığını biraz sulandırıyor.
- `business_adjusted_score_fraud_calibrated`: %1,86 → %5,89, en dik/monotonik olan. **Bu sürpriz değil:** bu fonksiyon zaten fraud oranına göre kalibre edildi, yani etiketle karşılaştırıldığında avantajlı çıkması `isFraud` kullanmasının doğal, "haksız" bir sonucu. Bu karşılaştırmanın asıl amacı da bunu göstermek: etiketsiz bir yöntemi etiketle kalibre edilmiş bir yöntemle "adil" biçimde kıyaslayamayız; ikincisi zaten cevabı biliyor.

**Sonuç:** hacim/güven tabanlı yöntem, projenin etiketsiz disiplinini koruyarak orta kovalarda gürültüyü azaltıyor (false-positive azaltma hedefine hizmet ediyor), ama en üst kovada fraud-oranı-kalibreli yöntem kadar keskin ayrım sağlamıyor. Bu, etiket kullanmamanın **gerçek bir maliyeti** olarak dürüstçe kabul ediliyor, gizlenmiyor.


## 2. Weekend Adjustment

Madde 1'deki `is_business_hours` tanımı zaten hafta sonunu "mesai dışı" içine katıyor
(`~is_weekend & 09:00-18:00`). O yüzden burada gerçekten **yeni** bir şey yakalamak istiyorsak,
önce hafta sonu etkisinin `hour_of_day`'den bağımsız, kendine ait bir sinyal olup olmadığını
göstermemiz gerekiyor; yoksa Madde 1'i farklı isimle tekrar etmiş oluruz.

**Kontrol:** aynı saat penceresine (09:00-18:00) hem hafta içi hem hafta sonu işlemlerini
kısıtlayıp fraud oranını karşılaştıralım. Saat sabit tutulduğunda hâlâ bir fark kalıyorsa, bu
`hour_of_day`'in yakalayamadığı, hafta sonuna özgü bağımsız bir etkidir.

In [9]:
weekday_mask = ~temporal["is_weekend_proxy"]
weekend_mask = temporal["is_weekend_proxy"]

print("Hacim, hafta içi vs hafta sonu:")
print(f"  hafta içi: {weekday_mask.sum():,}  |  hafta sonu: {weekend_mask.sum():,}")
print(f"  göreli hacim (hafta sonu / hafta içi): {weekend_mask.sum() / weekday_mask.sum():.3f}")
print()

check_full = temporal.merge(isfraud, on="TransactionID")
overall_rate = check_full.groupby("is_weekend_proxy")["isFraud"].mean() * 100
print("Fraud oranı (%), tüm gün, hafta içi vs hafta sonu:")
print(overall_rate)
print()

business_hour_mask = (check_full["hour_of_day"] >= 9) & (check_full["hour_of_day"] < 18)
controlled = check_full[business_hour_mask]
controlled_rate = controlled.groupby("is_weekend_proxy")["isFraud"].mean() * 100
print("Fraud oranı (%), SADECE 09:00-18:00 saatleri (saat sabitlenmiş), hafta içi vs hafta sonu:")
print(controlled_rate)

Hacim, hafta içi vs hafta sonu:
  hafta içi: 412,204  |  hafta sonu: 178,336
  göreli hacim (hafta sonu / hafta içi): 0.433



Fraud oranı (%), tüm gün, hafta içi vs hafta sonu:
is_weekend_proxy
False    3.432766
True     3.652095
Name: isFraud, dtype: float64

Fraud oranı (%), SADECE 09:00-18:00 saatleri (saat sabitlenmiş), hafta içi vs hafta sonu:
is_weekend_proxy
False    2.835899
True     3.160069
Name: isFraud, dtype: float64


**Sonuç: etki gerçek ve bağımsız, ama zayıf.** Saat penceresi 09:00-18:00'e sabitlendiğinde bile
hafta sonu fraud oranı hâlâ daha yüksek (\~%3,16 vs \~%2,84); bu, `hour_of_day`'in yakalamadığı,
hafta sonuna özgü ayrı bir sinyal. Ama Madde 1'in mesai-dışı etkisiyle kıyaslanınca çok daha
mütevazı: hacim farkı \~15 kat değil \~2,3 kat (178.336 / 412.204 ≈ 0,43); fraud oranı farkı \~%30
değil \~%6 (genel: %3,65 vs %3,43). Aşağıdaki iki yöntem bu zayıf sinyali **olduğu gibi**
yansıtacak; Madde 1'inki kadar dramatik görünmesi için abartılmayacak.

Aynı iki prensip, aynı gerekçeyle: hacim/güven tabanlı (etiketsiz) ve fraud-oranı-kalibreli
(`isFraud`'u yine bilinçli ve açıkça işaretlenmiş bir istisna olarak kullanan). İkisi de
`business_context.py`'nin `EXTREME_SCORE_PERCENTILE` (üst %1 asla hafifletilmez) ve
`MIN_CONFIDENCE_MULTIPLIER` (taban 0,5) sabitlerini yeniden kullanıyor; her bağlam boyutu için
ayrı bir eşik icat etmemek adına.

## Değerlendirme 1: Saatten Bağımsız (yalnızca hafta içi / hafta sonu)

`is_weekend_proxy` tek başına, saat bilgisi hiç kullanılmadan. Aşağıdaki iki yöntem her hafta sonu
saatini aynı muamele ile ele alıyor; bu değerlendirmenin bilerek kabul ettiği basitleştirme, bir
sonraki bölümde (Değerlendirme 2) gevşetilecek.

### Yöntem 1: Hacim/güven tabanlı (etiketsiz)

In [10]:
from src.services.anomaly.weekend_context import apply_weekend_confidence_adjustment

weekend_confidence_adjusted = apply_weekend_confidence_adjustment(final_raw, temporal)

print("Çarpanın aldığı benzersiz değerler:", sorted(weekend_confidence_adjusted["weekend_confidence_multiplier"].unique()))
print()
print("Hafta içi satırlarda çarpan hep 1,0 mı:", (weekend_confidence_adjusted.loc[~weekend_confidence_adjusted.is_weekend, "weekend_confidence_multiplier"] == 1.0).all())
print("Üst-%1'de olup hafta sonu olan, yine de muaf tutulan (çarpan=1,0) satır sayısı:",
      len(weekend_confidence_adjusted[weekend_confidence_adjusted.is_weekend & (weekend_confidence_adjusted["weekend_confidence_multiplier"] == 1.0)]))

Çarpanın aldığı benzersiz değerler: [np.float64(0.7163200745262055), np.float64(1.0)]

Hafta içi satırlarda çarpan hep 1,0 mı: True
Üst-%1'de olup hafta sonu olan, yine de muaf tutulan (çarpan=1,0) satır sayısı: 1817


Hafta içinde çarpan tam olarak 1,0: skor değişmiyor. Hafta sonunda tek bir sabit çarpan: 0,7163
(`0,5 + 0,5 × 0,4326`; göreli hacim doğrudan formüle giriyor), Madde 1'deki gibi saat-saat
değişen bir gradyan değil, çünkü burada zaten ikili (hafta içi/hafta sonu) bir ayrım var, saat
saat değişen bir hacim yok. Üst-%1'deki hafta sonu işlemleri (1.817 tanesi) yine muaf; aşırı uç
bir skor, sırf hafta sonuna denk geldi diye hafifletilmiyor.

### Yöntem 2: Fraud-oranı-kalibreli (`isFraud` kullanan, bilinçli istisna)

In [11]:
from src.services.anomaly.weekend_context import apply_weekend_fraud_calibrated_adjustment

weekend_fraud_adjusted = apply_weekend_fraud_calibrated_adjustment(final_raw, temporal, isfraud)

print("Çarpanın aldığı benzersiz değerler:", sorted(weekend_fraud_adjusted["weekend_fraud_calibration_multiplier"].unique()))
print("Hafta içi satırlarda çarpan hep 1,0 mı:", (weekend_fraud_adjusted.loc[~weekend_fraud_adjusted.is_weekend, "weekend_fraud_calibration_multiplier"] == 1.0).all())

Çarpanın aldığı benzersiz değerler: [np.float64(1.0), np.float64(1.0638926750568587)]
Hafta içi satırlarda çarpan hep 1,0 mı: True


Hafta sonu işlemleri \~%6,4 artırılıyor (1,0639); Madde 1'in fraud-kalibreli çarpanıyla (1,30)
kıyaslanınca çok daha ölçülü, çünkü kalibre ettiği fark da (%3,65 / %3,43) çok daha küçük. Yine
aynı istisna: bu adım `isFraud`'u bir tasarım parametresini belirlemek için kullanıyor, sadece
sonradan doğrulamak için değil; gerçek bir üretim sisteminde bu tür bir kalibrasyonun kendi
tutma-seti (held-out) doğrulamasından geçmesi gerekir, bu kapsamın dışında.

### Doğrulama 1: üst-%1 ve üst-%5'te ne değişiyor

In [12]:
weekend_check = final_raw.merge(
    weekend_confidence_adjusted[["TransactionID", "weekend_adjusted_score_confidence_based"]], on="TransactionID"
).merge(
    weekend_fraud_adjusted[["TransactionID", "weekend_adjusted_score_fraud_calibrated"]], on="TransactionID"
)

for pct, label in [(0.01, "üst-%1"), (0.05, "üst-%5")]:
    raw_top = set(weekend_check.loc[weekend_check["final_raw_anomaly_score"] >= weekend_check["final_raw_anomaly_score"].quantile(1 - pct), "TransactionID"])
    conf_top = set(weekend_check.loc[weekend_check["weekend_adjusted_score_confidence_based"] >= weekend_check["weekend_adjusted_score_confidence_based"].quantile(1 - pct), "TransactionID"])
    fraud_top = set(weekend_check.loc[weekend_check["weekend_adjusted_score_fraud_calibrated"] >= weekend_check["weekend_adjusted_score_fraud_calibrated"].quantile(1 - pct), "TransactionID"])

    print(f"--- {label} (küme boyutu: {len(raw_top):,}) ---")
    print(f"  ham vs hacim/güven-tabanlı: {len(raw_top & conf_top):,} ortak ({len(raw_top & conf_top) / len(raw_top):.1%} korunuyor)")
    print(f"  ham vs fraud-kalibreli:     {len(raw_top & fraud_top):,} ortak ({len(raw_top & fraud_top) / len(raw_top):.1%} korunuyor)")
    print()

--- üst-%1 (küme boyutu: 5,906) ---
  ham vs hacim/güven-tabanlı: 5,906 ortak (100.0% korunuyor)
  ham vs fraud-kalibreli:     3,479 ortak (58.9% korunuyor)



--- üst-%5 (küme boyutu: 29,527) ---
  ham vs hacim/güven-tabanlı: 21,849 ortak (74.0% korunuyor)
  ham vs fraud-kalibreli:     25,362 ortak (85.9% korunuyor)



**Üst-%1 tam olarak korunuyor (hacim/güven-tabanlı için %100): beklenen, çünkü üst-%1 eşiği ile
`EXTREME_SCORE_PERCENTILE` aynı eşik (0,99); üst-%1'deki her işlem zaten muafiyet kapsamında,
hafta sonu olsa bile hafifletilmiyor.** Fraud-kalibreli yöntemde üst-%1 bile değişiyor (\~%59
korunuyor) çünkü o yöntemde muafiyet yok; hafta sonu her işlem artırılıyor, bu da eşiğin
kendisini oynatıyor. Üst-%5'te ikisi de değişiyor (hacim/güven: \~%74 korunuyor, fraud-kalibreli:
\~%86 korunuyor); mekanizma tutarlı, sadece Madde 1'den çok daha hafif bir etki bırakıyor, önceden
beklendiği gibi.

### Betimleyici kontrol: fraud oranıyla hizalanma, kovalara göre

In [13]:
weekend_check_fraud = weekend_check.merge(isfraud, on="TransactionID")

for col in ["final_raw_anomaly_score", "weekend_adjusted_score_confidence_based", "weekend_adjusted_score_fraud_calibrated"]:
    weekend_check_fraud["decile"] = pd.qcut(weekend_check_fraud[col], 10, labels=False, duplicates="drop") + 1
    rates = weekend_check_fraud.groupby("decile")["isFraud"].mean() * 100
    print(col)
    print({k: round(v, 2) for k, v in rates.items()})
    print()

final_raw_anomaly_score
{1: 1.82, 2: 2.01, 3: 2.16, 4: 2.69, 5: 3.08, 6: 3.51, 7: 4.29, 8: 4.51, 9: 5.57, 10: 5.34}



weekend_adjusted_score_confidence_based
{1: 1.82, 2: 2.13, 3: 2.43, 4: 2.63, 5: 3.36, 6: 3.65, 7: 4.27, 8: 4.47, 9: 4.81, 10: 5.42}



weekend_adjusted_score_fraud_calibrated
{1: 1.86, 2: 1.95, 3: 2.18, 4: 2.72, 5: 3.04, 6: 3.53, 7: 4.18, 8: 4.58, 9: 5.54, 10: 5.41}



**Küçük ama tutarlı bir iyileşme, hacim/güven tabanlı yöntemde bile:** ham skorda 9. ve 10. kova
arasında hafif bir ters dönüş var (%5,57 → %5,34: monotonik değil). Hacim/güven-tabanlı düzeltme
bunu düzeltiyor (9. kova %4,81 → 10. kova %5,42, artık monotonik); ve bunu **hiç `isFraud`
kullanmadan** başarıyor, sadece hafta sonu işlemlerin bir kısmını orta kovalara doğru kaydırarak.
Fraud-kalibreli yöntemde de aynı ters dönüş büyük ölçüde düzeliyor (%5,54 → %5,41; hâlâ hafif bir
fark var ama ham skordan çok daha yakın). Madde 1'deki gibi keskin bir üst-kova ayrımı yok (etki
zaten küçük), ama ikisi de yönü doğru: false-positive'e en çok benzeyen köşe biraz temizleniyor.



## Değerlendirme 2: Hafta Sonu Yoğun Saatler (14:00–01:59)

Değerlendirme 1, her hafta sonu saatini aynı kefeye koyuyordu. Ama saatlik hacim kırılımına
bakınca hafta sonu kendi içinde düz değil: 14:00–01:59 arası sürekli yüksek bir plato (>10.000
işlem/saat), geri kalan saatler (02:00–13:59) belirgin şekilde daha seyrek; dip nokta saat
09:00'da (720 işlem). Bu ayrımı `WEEKEND_PEAK_HOUR_START=14` / `WEEKEND_PEAK_HOUR_END=2` olarak
sabitleyip (business hours'ın 09-18 sınırı gibi, veriden türetilen ama sabit bir politika kuralı),
hafta sonunu kendi içinde yoğun/düşük diye ikiye bölerek aynı iki yöntemi tekrar uyguluyoruz;
bu sefer hafta içi ve hafta sonu-yoğun saatler dokunulmadan (çarpan=1,0), sadece hafta sonu-düşük
saatler düzeltiliyor.

In [14]:
from src.services.anomaly.weekend_context import WEEKEND_PEAK_HOUR_START, WEEKEND_PEAK_HOUR_END

in_peak_hour = (temporal["hour_of_day"] >= WEEKEND_PEAK_HOUR_START) | (temporal["hour_of_day"] < WEEKEND_PEAK_HOUR_END)

check2 = temporal.merge(isfraud, on="TransactionID")
check2["in_peak_hour"] = in_peak_hour.to_numpy()

table = check2.groupby(["is_weekend_proxy", "in_peak_hour"])["isFraud"].agg(["mean", "count"])
table["mean"] = (table["mean"] * 100).round(3)
table.index.names = ["is_weekend", "is_peak_window(14-02)"]
print(table)


                                   mean   count
is_weekend is_peak_window(14-02)               
False      False                  4.289   92751
           True                   3.184  319453
True       False                  5.088   37324
           True                   3.272  141012


**2x2 tablo, düşük hacimli hücrenin ne kadar keskin öne çıktığını gösteriyor.** Fraud oranını
asıl süren şey saat penceresi (yoğun/düşük); ama hafta sonu bunun üstüne ek bir artış katıyor, ve
bu artış düşük-hacim penceresinde çok daha belirgin (hafta sonu düşük: %5,09 vs hafta içi düşük:
%4,29, fark +0,80 puan) yoğun pencerede ise neredeyse yok (hafta sonu yoğun: %3,27 vs hafta içi
yoğun: %3,18, fark +0,09 puan). Yani "hafta sonu etkisi" büyük ölçüde hafta sonunun **kendi düşük
saatlerinde** yoğunlaşıyor; bu yüzden düzeltmeyi de sadece o dilime (hafta sonu + düşük saat)
uyguluyoruz, tüm hafta sonuna değil.

### Yöntem 1: Hacim/güven tabanlı (etiketsiz)

In [15]:
from src.services.anomaly.weekend_context import apply_weekend_peak_confidence_adjustment

weekend_peak_confidence_adjusted = apply_weekend_peak_confidence_adjustment(final_raw, temporal)

print("Çarpanın aldığı benzersiz değerler:", sorted(weekend_peak_confidence_adjusted["weekend_peak_confidence_multiplier"].unique()))
print("Hafta sonu-düşük olmayan satırlarda çarpan hep 1,0 mı:",
      (weekend_peak_confidence_adjusted.loc[~weekend_peak_confidence_adjusted.is_weekend_off_peak, "weekend_peak_confidence_multiplier"] == 1.0).all())
print("Etkilenen satır sayısı (hafta sonu + düşük saat):", weekend_peak_confidence_adjusted["is_weekend_off_peak"].sum())

Çarpanın aldığı benzersiz değerler: [np.float64(0.632343346665532), np.float64(1.0)]
Hafta sonu-düşük olmayan satırlarda çarpan hep 1,0 mı: True
Etkilenen satır sayısı (hafta sonu + düşük saat): 37324


Çarpan 0,6323: Değerlendirme 1'in tek çarpanından (0,7163) belirgin şekilde daha güçlü bir
hafifletme, çünkü bu sefer kıyaslanan dilim (hafta sonu-düşük, 37.324 işlem) hafta sonu-yoğun'a
(141.012 işlem) göre çok daha seyrek; göreli hacim 0,265, önceki 0,433'ten daha küçük. Hafta
içi ve hafta sonu-yoğun saatler tamamen dokunulmuyor.

### Yöntem 2: Fraud-oranı-kalibreli (`isFraud` kullanan, bilinçli istisna)

In [16]:
from src.services.anomaly.weekend_context import apply_weekend_peak_fraud_calibrated_adjustment

weekend_peak_fraud_adjusted = apply_weekend_peak_fraud_calibrated_adjustment(final_raw, temporal, isfraud)

print("Çarpanın aldığı benzersiz değerler:", sorted(weekend_peak_fraud_adjusted["weekend_peak_fraud_calibration_multiplier"].unique()))
print("Hafta sonu-düşük olmayan satırlarda çarpan hep 1,0 mı:",
      (weekend_peak_fraud_adjusted.loc[~weekend_peak_fraud_adjusted.is_weekend_off_peak, "weekend_peak_fraud_calibration_multiplier"] == 1.0).all())

Çarpanın aldığı benzersiz değerler: [np.float64(1.0), np.float64(1.5549458375182688)]
Hafta sonu-düşük olmayan satırlarda çarpan hep 1,0 mı: True


Çarpan 1,5549: Değerlendirme 1'in çarpanından (1,0639) çok daha güçlü bir artırma. Beklenen:
buradaki fraud-oranı kontrastı (%5,09 / %3,27) Değerlendirme 1'inkinden (%3,65 / %3,43) çok daha
keskin, çünkü hafta sonu etkisini saatten bağımsız ölçmek onu düşük-hacim saatleriyle
seyreltiyordu; saate göre ayırınca asıl sinyal ortaya çıkıyor.

### Doğrulama: üst-%1 ve üst-%5'te ne değişiyor

In [17]:
weekend_peak_check = weekend_check.merge(
    weekend_peak_confidence_adjusted[["TransactionID", "weekend_peak_adjusted_score_confidence_based"]], on="TransactionID"
).merge(
    weekend_peak_fraud_adjusted[["TransactionID", "weekend_peak_adjusted_score_fraud_calibrated"]], on="TransactionID"
)

cols = [
    "weekend_adjusted_score_confidence_based", "weekend_adjusted_score_fraud_calibrated",
    "weekend_peak_adjusted_score_confidence_based", "weekend_peak_adjusted_score_fraud_calibrated",
]

for pct, label in [(0.01, "üst-%1"), (0.05, "üst-%5")]:
    raw_top = set(weekend_peak_check.loc[weekend_peak_check["final_raw_anomaly_score"] >= weekend_peak_check["final_raw_anomaly_score"].quantile(1 - pct), "TransactionID"])
    print(f"--- {label} (küme boyutu: {len(raw_top):,}) ---")
    for c in cols:
        t = set(weekend_peak_check.loc[weekend_peak_check[c] >= weekend_peak_check[c].quantile(1 - pct), "TransactionID"])
        print(f"  {c}: {len(raw_top & t):,} ortak ({len(raw_top & t) / len(raw_top):.1%})")
    print()

--- üst-%1 (küme boyutu: 5,906) ---
  weekend_adjusted_score_confidence_based: 5,906 ortak (100.0%)
  weekend_adjusted_score_fraud_calibrated: 3,479 ortak (58.9%)
  weekend_peak_adjusted_score_confidence_based: 5,906 ortak (100.0%)


  weekend_peak_adjusted_score_fraud_calibrated: 330 ortak (5.6%)

--- üst-%5 (küme boyutu: 29,527) ---
  weekend_adjusted_score_confidence_based: 21,849 ortak (74.0%)


  weekend_adjusted_score_fraud_calibrated: 25,362 ortak (85.9%)


  weekend_peak_adjusted_score_confidence_based: 27,877 ortak (94.4%)
  weekend_peak_adjusted_score_fraud_calibrated: 16,539 ortak (56.0%)



**Hacim/güven tabanlı yöntemde üst-%1 hâlâ tam korunuyor** (aynı `EXTREME_SCORE_PERCENTILE`
muafiyeti), üst-%5'te ise Değerlendirme 1'den (%74,0) daha az bozuluyor (%94,4 korunuyor); daha
dar bir dilimi (37.324 vs 178.336 işlem) etkilediği için beklenen bir sonuç.

**Fraud-kalibreli yöntemde tam tersi oluyor: üst-%1 sadece %5,6 korunuyor** (Değerlendirme 1'de
%58,9'du); 1,5549'luk güçlü artırma, dar ama yoğun bir dilimi (37.324 işlem) o kadar yukarı
itiyor ki üst-%1'in bileşimi neredeyse tamamen değişiyor. Bu, etiket-kalibreli bir düzeltmenin ne
kadar agresif olabileceğinin somut bir göstergesi; dar bir dilime güçlü bir çarpan uygulamak,
geniş bir dilime zayıf bir çarpan uygulamaktan çok daha fazla sıralama değişikliği yaratabiliyor.

### Betimleyici kontrol: iki değerlendirmeyi kova bazında karşılaştır

In [18]:
weekend_peak_check_fraud = weekend_peak_check.merge(isfraud, on="TransactionID")

all_cols = ["final_raw_anomaly_score"] + cols
for col in all_cols:
    weekend_peak_check_fraud["decile"] = pd.qcut(weekend_peak_check_fraud[col], 10, labels=False, duplicates="drop") + 1
    rates = weekend_peak_check_fraud.groupby("decile")["isFraud"].mean() * 100
    print(col)
    print({k: round(v, 2) for k, v in rates.items()})
    print()

final_raw_anomaly_score
{1: 1.82, 2: 2.01, 3: 2.16, 4: 2.69, 5: 3.08, 6: 3.51, 7: 4.29, 8: 4.51, 9: 5.57, 10: 5.34}



weekend_adjusted_score_confidence_based
{1: 1.82, 2: 2.13, 3: 2.43, 4: 2.63, 5: 3.36, 6: 3.65, 7: 4.27, 8: 4.47, 9: 4.81, 10: 5.42}



weekend_adjusted_score_fraud_calibrated
{1: 1.86, 2: 1.95, 3: 2.18, 4: 2.72, 5: 3.04, 6: 3.53, 7: 4.18, 8: 4.58, 9: 5.54, 10: 5.41}



weekend_peak_adjusted_score_confidence_based
{1: 1.89, 2: 2.22, 3: 2.43, 4: 2.74, 5: 3.21, 6: 3.57, 7: 4.24, 8: 4.28, 9: 5.26, 10: 5.16}



weekend_peak_adjusted_score_fraud_calibrated
{1: 1.84, 2: 1.98, 3: 2.16, 4: 2.64, 5: 3.0, 6: 3.59, 7: 4.19, 8: 4.48, 9: 5.48, 10: 5.62}



**Peak-farkındalıklı fraud-kalibreli yöntem, en iyi üst-kova ayrımını veriyor:** 9. kova %5,48 →
10. kova %5,62; hem ham skordaki ters dönüşü (%5,57→%5,34) düzeltiyor hem de Değerlendirme 1'in
fraud-kalibreli versiyonundan (%5,54→%5,41) daha keskin bir artış bırakıyor. Bunun sebebi açık: bu
yöntem daha keskin bir fraud-oranı kontrastından (1,5549) besleniyor.

**Peak-farkındalıklı hacim/güven yönteminde ise durum tam tersi, beklenmedik bir sonuç:** 9. kova
%5,26 → 10. kova %5,16, hâlâ hafif bir ters dönüş var, üstelik Değerlendirme 1'in hacim/güven
versiyonundan (%4,81→%5,42, tam monotonik) daha kötü. Bunu gizlemek yerine açıkça not ediyoruz:
daha dar bir dilimi (sadece hafta sonu-düşük, 37.324 işlem) daha güçlü hafifletmek (0,6323 vs
0,7163), o dilimin İÇİNDEKİ gerçek anomalileri de aynı oranda bastırıyor. "Dar dilim + güçlü
hafifletme" kombinasyonu, "geniş dilim + ölçülü hafifletme"den daha fazla gerçek sinyali de
birlikte götürebiliyor. Etiketsiz bir yöntemi daha "isabetli" yapmak için daralttıkça, isabetsiz
olma riski de artabiliyor; bu, false-positive azaltımı ile false-negative riski arasındaki
klasik ödünleşimin küçük bir örneği.



## 3. Trusted Entity Adjustment

"Güvenilir kullanıcı/entity" için en doğal aday, Case 3'ün zaten ürettiği
`user_transaction_count_so_far` (bu karta ait, bu işlemden önceki toplam işlem sayısı, `card1`
bazında). Naif beklenti: uzun geçmişi olan bir kart daha "tanıdık", yeni bir karttan daha az
şüpheli olmalı. Önce bunu veriyle test edelim.

In [19]:
from src.services.features.entity import build_entity_features

entity = build_entity_features(parquet_path)
trust_check = entity.merge(isfraud, on="TransactionID")

bins = [-1, 0, 1, 2, 5, 10, 20, 50, 100, 1000, 100000]
labels = ["0 (ilk işlem)", "1", "2", "3-5", "6-10", "11-20", "21-50", "51-100", "101-1000", "1000+"]
trust_check["bucket"] = pd.cut(trust_check["user_transaction_count_so_far"], bins=bins, labels=labels)

bucket_table = trust_check.groupby("bucket", observed=True)["isFraud"].agg(["mean", "count"])
bucket_table["mean"] = (bucket_table["mean"] * 100).round(3)
print(bucket_table)

                mean   count
bucket                      
0 (ilk işlem)  2.568   13553
1              2.483   10109
2              2.411    8419
3-5            2.551   19717
6-10           2.582   23276
11-20          2.734   30906
21-50          3.110   51285
51-100         3.513   45608
101-1000       3.910  207649
1000+          3.664  180018


**Naif beklentinin tam tersi.** Fraud oranı en düşük yeni/az-geçmişli kartlarda (%2,4-2,6), geçmiş
işlem sayısı arttıkça YÜKSELİYOR, 101-1000 bandında zirve yapıyor (%3,91), 1000+ bandında hafif
geriliyor ama yine de düşük-geçmiş grubunun üzerinde (%3,66) kalıyor.

Muhtemel sebep: `card1`, bu ölçekte gerçek tek bir müşteriyi değil, yüksek hacimde paylaşılan bir
"bucket" değerini temsil ediyor olabilir (Case 1/3'te de karşılaşılan bir örüntü); binlerce işlemi
olan bir `card1` değeri muhtemelen birçok farklı gerçek kartın paylaştığı bir grup, tek bir sadık
müşteri değil. Yani "geçmiş işlem sayısı" burada bireysel güvenilirliği değil, o bucket'ın ne kadar
genel/paylaşılan olduğunu ölçüyor olabilir.

**Not (saklanmıyor):** ilişki tepe noktasından sonra tam monotonik değil; üst-%10'a (>=3700 önceki
işlem) kısıtlanınca oran kısmen tersine dönüyor (%3,24 vs geri kalanın %3,53'ü). Aşağıdaki
`TRUST_THRESHOLD=100` sınırı, yükselişin büyük kısmını taşıyan bölgeyi yakalayan, 50-200 arası
komşu eşiklerde tutarlı kalan (hepsi \~1,23-1,36x aynı yönde) ama tek geçerli seçim olmayan bir
politika kararı.

### Yöntem 1: Tenure tabanlı, "şüpheden yararlanma" (etiketsiz, naif yön)

In [20]:
from src.services.anomaly.trusted_entity_context import apply_trust_confidence_adjustment

trust_confidence_adjusted = apply_trust_confidence_adjustment(final_raw, entity)

print("Çarpanın aldığı benzersiz değerler:", sorted(trust_confidence_adjusted["trust_confidence_multiplier"].unique()))
print("Güvenilmeyen (yeni/az-geçmiş) satırlarda çarpan hep 1,0 mı:",
      (trust_confidence_adjusted.loc[~trust_confidence_adjusted.is_trusted_entity, "trust_confidence_multiplier"] == 1.0).all())
print("Güvenilir sayılan işlem oranı:", f'{trust_confidence_adjusted["is_trusted_entity"].mean():.1%}')

Çarpanın aldığı benzersiz değerler: [np.float64(0.5), np.float64(1.0)]
Güvenilmeyen (yeni/az-geçmiş) satırlarda çarpan hep 1,0 mı: True
Güvenilir sayılan işlem oranı: 65.8%


Bu, projedeki diğer güven-çarpanlarından farklı bir mantık: business_context.py ve
weekend_context.py'de SEYREK/gürültülü taraf hafifletiliyordu (istatistiksel gerekçe). Burada ise
tam tersi: YERLEŞİK (>=100 önceki işlem) tarafı hafifletiyoruz, çünkü "güvenilir entity" kuralı
istatistiksel gürültü değil, operasyonel bir "tanıdık müşteriye şüpheden yararlan" politikası.
İşlemlerin %65,8'i bu tanıma göre "güvenilir" sayılıyor; büyük bir çoğunluk, dampinglenen kitle
de büyük.

### Yöntem 2: Fraud-oranı-kalibreli (`isFraud` kullanan, bilinçli istisna)

In [21]:
from src.services.anomaly.trusted_entity_context import apply_trust_fraud_calibrated_adjustment

trust_fraud_adjusted = apply_trust_fraud_calibrated_adjustment(final_raw, entity, isfraud)

print("Çarpanın aldığı benzersiz değerler:", sorted(trust_fraud_adjusted["trust_fraud_calibration_multiplier"].unique()))
print("Güvenilmeyen satırlarda çarpan hep 1,0 mı:",
      (trust_fraud_adjusted.loc[~trust_fraud_adjusted.is_trusted_entity, "trust_fraud_calibration_multiplier"] == 1.0).all())

Çarpanın aldığı benzersiz değerler: [np.float64(1.0), np.float64(1.2959549885982842)]
Güvenilmeyen satırlarda çarpan hep 1,0 mı: True


Çarpan 1,2960: Yöntem 1'in tam tersi yönde: "güvenilir" (yerleşik) kartlar hafifletilmek yerine
**artırılıyor**, çünkü gerçek ölçülen fraud oranı bu grupta daha yüksek. business_context.py'nin
mesai-dışı bulgusuyla aynı örüntü: naif/operasyonel yön ile veri-kalibreli yön burada da zıt.

### Doğrulama: üst-%1 ve üst-%5'te ne değişiyor

In [22]:
trust_check_scores = final_raw.merge(
    trust_confidence_adjusted[["TransactionID", "trust_adjusted_score_confidence_based"]], on="TransactionID"
).merge(
    trust_fraud_adjusted[["TransactionID", "trust_adjusted_score_fraud_calibrated"]], on="TransactionID"
)

for pct, label in [(0.01, "üst-%1"), (0.05, "üst-%5")]:
    raw_top = set(trust_check_scores.loc[trust_check_scores["final_raw_anomaly_score"] >= trust_check_scores["final_raw_anomaly_score"].quantile(1 - pct), "TransactionID"])
    print(f"--- {label} (küme boyutu: {len(raw_top):,}) ---")
    for c in ["trust_adjusted_score_confidence_based", "trust_adjusted_score_fraud_calibrated"]:
        t = set(trust_check_scores.loc[trust_check_scores[c] >= trust_check_scores[c].quantile(1 - pct), "TransactionID"])
        print(f"  {c}: {len(raw_top & t):,} ortak ({len(raw_top & t) / len(raw_top):.1%})")
    print()

--- üst-%1 (küme boyutu: 5,906) ---
  trust_adjusted_score_confidence_based: 5,906 ortak (100.0%)
  trust_adjusted_score_fraud_calibrated: 4,114 ortak (69.7%)

--- üst-%5 (küme boyutu: 29,527) ---
  trust_adjusted_score_confidence_based: 15,289 ortak (51.8%)
  trust_adjusted_score_fraud_calibrated: 18,352 ortak (62.2%)



Üst-%1'de hacim/güven-tabanlı yöntem yine tam korunuyor (`EXTREME_SCORE_PERCENTILE` muafiyeti),
fraud-kalibreli yöntemde ise (\~%69,7 korunuyor); bu iki değerlendirmedeki (Madde 1/2) fraud-kalibreli
sonuçlarla tutarlı bir örüntü. Üst-%5'te iki yöntem de belirgin biçimde karışıyor (\~%52 ve \~%62);
çünkü "güvenilir" grup, işlemlerin üçte ikisini kapsıyor, bu da geniş bir kısmın sırasını
oynatabiliyor.

### Betimleyici kontrol: fraud oranıyla hizalanma, kovalara göre

In [23]:
trust_check_fraud = trust_check_scores.merge(isfraud, on="TransactionID")

for col in ["final_raw_anomaly_score", "trust_adjusted_score_confidence_based", "trust_adjusted_score_fraud_calibrated"]:
    trust_check_fraud["decile"] = pd.qcut(trust_check_fraud[col], 10, labels=False, duplicates="drop") + 1
    rates = trust_check_fraud.groupby("decile")["isFraud"].mean() * 100
    print(col)
    print({k: round(v, 2) for k, v in rates.items()})
    print()

final_raw_anomaly_score
{1: 1.82, 2: 2.01, 3: 2.16, 4: 2.69, 5: 3.08, 6: 3.51, 7: 4.29, 8: 4.51, 9: 5.57, 10: 5.34}

trust_adjusted_score_confidence_based
{1: 1.93, 2: 2.21, 3: 2.64, 4: 3.27, 5: 4.06, 6: 4.22, 7: 5.14, 8: 4.21, 9: 3.2, 10: 4.11}



trust_adjusted_score_fraud_calibrated
{1: 1.76, 2: 1.95, 3: 2.27, 4: 2.52, 5: 2.99, 6: 3.54, 7: 3.92, 8: 4.69, 9: 5.32, 10: 6.02}



**En çarpıcı sonuç şu ana kadarki tüm Case 6 karşılaştırmalarında bu oldu.** Tenure-tabanlı, naif
yöndeki Yöntem 1 kova yapısını *bozuyor*: {1: 1,93 ... 7: 5,14, 8: 4,21, 9: **3,20**, 10: 4,11}.
9. kova aniden %3,20'ye düşüyor, ham skordaki hafif dalgalanmadan (9→10: %5,57→%5,34) çok daha
ciddi bir düzensizlik. Sebep açık: işlemlerin %65,8'i (ve gerçekte bu grubun fraud oranı daha
yüksek) yarı yarıya hafifletiliyor, bu da yüksek-skorlu işlemleri orta kovalara doğru itip kova
sırasını gerçek anlamda bozuyor. "Yanlış yönde" bir düzeltmenin sadece işe yaramaması değil,
**aktif olarak zarar verebileceğinin** somut bir kanıtı.

Fraud-kalibreli Yöntem 2 ise tam tersi: {1: 1,76 ... 9: 5,32, 10: **6,02**}, tamamen monotonik,
ve bu projede üretilen HERHANGİ bir skor varyantı arasında en temiz üst-kova ayrımı (10. kova
%6,02, ham skorun %5,34'ünden belirgin şekilde daha yüksek). Beklenen bir avantaj (etiketle kalibre
edildi), ama büyüklüğü yine de not edilmeye değer.

**Sonuç:** bu madde, projenin "isFraud kullanmayı yalnızca doğrulama için sakla" disiplininin
neden önemli olduğunun en net örneğini verdi. Operasyonel açıdan makul görünen bir kural
("uzun geçmiş = güven"), bu özel veri setinde test edilmeden uygulansaydı, false-positive'i
azaltmak yerine tam tersini yapıp gerçek anlamda zarar verirdi.



## 4. False Positive Azaltımı: Ölçüm

Şimdiye kadar her madde kendi üst-%1/%5 kümesinin ne kadar değiştiğini gösterdi, ama hiçbiri
doğrudan "false positive azaltıldı mı" sorusuna cevap vermedi. Bunu yapmanın en dürüst yolu: sabit
bir **alert bütçesi** tanımlamak (örn. skor sırasına göre en yüksek %5'i "şüpheli" diye işaretle;
gerçek bir operasyon ekibinin günlük inceleyebileceği sabit sayıda işlem gibi düşünülebilir), ve bu
bütçe sabitken hangi skorun daha AZ false positive (gerçekte fraud olmayan ama işaretlenen) ve daha
ÇOK true positive (gerçekte fraud olan ve işaretlenen) ürettiğini karşılaştırmak.

Bu tamamen betimleyici bir ölçüm; `isFraud` burada hiçbir tasarım kararını yönlendirmiyor, sadece
zaten kurulmuş 8 context-adjustment'ın (Madde 1-3'ün etiketsiz ve etiket-kalibreli versiyonları)
gerçek etkisini sonradan doğruluyor. Projenin baştan beri sürdürdüğü disiplinle tam uyumlu.

In [24]:
fp_base = final_raw.merge(isfraud, on="TransactionID") \
    .merge(confidence_adjusted[["TransactionID", "business_adjusted_score_confidence_based"]], on="TransactionID") \
    .merge(fraud_calibrated[["TransactionID", "business_adjusted_score_fraud_calibrated"]], on="TransactionID") \
    .merge(weekend_confidence_adjusted[["TransactionID", "weekend_adjusted_score_confidence_based"]], on="TransactionID") \
    .merge(weekend_fraud_adjusted[["TransactionID", "weekend_adjusted_score_fraud_calibrated"]], on="TransactionID") \
    .merge(weekend_peak_confidence_adjusted[["TransactionID", "weekend_peak_adjusted_score_confidence_based"]], on="TransactionID") \
    .merge(weekend_peak_fraud_adjusted[["TransactionID", "weekend_peak_adjusted_score_fraud_calibrated"]], on="TransactionID") \
    .merge(trust_confidence_adjusted[["TransactionID", "trust_adjusted_score_confidence_based"]], on="TransactionID") \
    .merge(trust_fraud_adjusted[["TransactionID", "trust_adjusted_score_fraud_calibrated"]], on="TransactionID")

score_variants = [
    "final_raw_anomaly_score",
    "business_adjusted_score_confidence_based", "business_adjusted_score_fraud_calibrated",
    "weekend_adjusted_score_confidence_based", "weekend_adjusted_score_fraud_calibrated",
    "weekend_peak_adjusted_score_confidence_based", "weekend_peak_adjusted_score_fraud_calibrated",
    "trust_adjusted_score_confidence_based", "trust_adjusted_score_fraud_calibrated",
]

total_fraud = fp_base["isFraud"].sum()


def fp_reduction_table(budget_pct):
    budget = round(len(fp_base) * budget_pct)
    rows = []
    raw_fp = None
    for col in score_variants:
        flagged = fp_base.nlargest(budget, col)
        tp = int(flagged["isFraud"].sum())
        fp = budget - tp
        if col == "final_raw_anomaly_score":
            raw_fp = fp
        rows.append({
            "score": col, "TP": tp, "FP": fp,
            "precision_%": round(tp / budget * 100, 2),
            "recall_%": round(tp / total_fraud * 100, 2),
            "FP_degisim": fp - raw_fp,
        })
    return pd.DataFrame(rows).set_index("score")


print(f"Alert bütçesi: üst-%5 ({round(len(fp_base)*0.05):,} işlem)")
table_5pct = fp_reduction_table(0.05)
print(table_5pct)

Alert bütçesi: üst-%5 (29,527 işlem)


                                                TP     FP  precision_%  \
score                                                                    
final_raw_anomaly_score                       1481  28046         5.02   
business_adjusted_score_confidence_based      1369  28158         4.64   
business_adjusted_score_fraud_calibrated      1575  27952         5.33   
weekend_adjusted_score_confidence_based       1497  28030         5.07   
weekend_adjusted_score_fraud_calibrated       1465  28062         4.96   
weekend_peak_adjusted_score_confidence_based  1454  28073         4.92   
weekend_peak_adjusted_score_fraud_calibrated  1761  27766         5.96   
trust_adjusted_score_confidence_based         1278  28249         4.33   
trust_adjusted_score_fraud_calibrated         1683  27844         5.70   

                                              recall_%  FP_degisim  
score                                                               
final_raw_anomaly_score                        

**Karışık bir tablo, ve bu, gizlenmemesi gereken en önemli bulgu.** Dört etiketsiz (confidence-based)
kuraldan sadece BİRİ (`weekend_adjusted_score_confidence_based`) gerçekten false positive'i azaltıyor
(-16). Diğer üçü (`business`, `weekend_peak`, `trust`) aslında **daha FAZLA** false positive üretiyor
(+112, +27, +203). "Mantıklı görünen" bir iş kuralı, test edilmeden uygulansaydı izleme kalitesini
kötüleştirirdi. `trust_adjusted_score_confidence_based` en kötü performansı veriyor; Madde 3'teki kova
bozulmasıyla birebir tutarlı.

Dört etiket-kalibreli (fraud-calibrated) kuralın **hepsi** false positive'i azaltıyor (-94, -16→+16
hariç weekend_fraud da hafif kötü, -280, -202); beklenen, çünkü zaten etikete göre kalibre edildiler;
bu karşılaştırmanın kendisi zaten adil değil (cevabı bilen bir yöntemi bilmeyenle kıyaslıyoruz), ama
yine de en güçlü azaltım `weekend_peak_adjusted_score_fraud_calibrated`'dan geliyor (-280 FP, precision
%5,02→%5,96); Madde 2'nin en keskin fraud-oranı kontrastına sahip olmasıyla tutarlı.

In [25]:
print(f"Alert bütçesi: üst-%1 ({round(len(fp_base)*0.01):,} işlem)")
table_1pct = fp_reduction_table(0.01)
print(table_1pct)

Alert bütçesi: üst-%1 (5,905 işlem)


                                               TP    FP  precision_%  \
score                                                                  
final_raw_anomaly_score                       166  5739         2.81   
business_adjusted_score_confidence_based      166  5739         2.81   
business_adjusted_score_fraud_calibrated      226  5679         3.83   
weekend_adjusted_score_confidence_based       166  5739         2.81   
weekend_adjusted_score_fraud_calibrated       241  5664         4.08   
weekend_peak_adjusted_score_confidence_based  166  5739         2.81   
weekend_peak_adjusted_score_fraud_calibrated  482  5423         8.16   
trust_adjusted_score_confidence_based         166  5739         2.81   
trust_adjusted_score_fraud_calibrated         229  5676         3.88   

                                              recall_%  FP_degisim  
score                                                               
final_raw_anomaly_score                           0.80           0  


**Üst-%1'de tüm etiketsiz kurallar FP_değişim=0 veriyor: hata değil, tasarım gereği.** Her üç
modülün de paylaştığı `EXTREME_SCORE_PERCENTILE=0.99` muafiyeti, üst-%1'i hiç değiştirmeden bırakıyor
(zaten en yüksek skorlu işlemler hiçbir zaman hafifletilmiyor); Madde 1-3'te ayrı ayrı gözlemlenen bu
davranış, burada da birebir tekrarlanıyor, tutarlılığı doğruluyor. Etiket-kalibreli kurallar burada da
false positive'i azaltıyor, en güçlüsü yine `weekend_peak` (precision %2,81→%8,16; dar dilime güçlü
bir çarpan uygulamanın, doğru yönde kalibre edildiğinde ne kadar etkili olabileceğinin göstergesi).



## 5. Coğrafi Risk (Geographic Risk)

Bu madde için üç aday sinyal test edildi. `addr2` (billing bölge/ülke kodu) şimdiye kadarki EN
GÜÇLÜ context sinyali çıktı; `dist1` (mesafe) ve `P_emaildomain`'in ülke-kodu benzeri uzantıları
(uk/de/fr/mx/es/jp) test edildi ama işe yaramadı; bunlar da dürüstçe gösterilecek, sadece
kullanılan sinyal değil.

In [26]:
raw_geo = pq.ParquetFile(parquet_path).read(columns=["TransactionID", "addr2", "dist1", "P_emaildomain"]).to_pandas()
geo_check = raw_geo.merge(isfraud, on="TransactionID")

is_domestic = geo_check["addr2"] == 87
is_foreign = geo_check["addr2"].notna() & (geo_check["addr2"] != 87)
is_missing = geo_check["addr2"].isna()

print("addr2, yerli/yabancı/eksik dağılımı ve fraud oranı:")
for name, mask in [("yerli (87)", is_domestic), ("yabancı (≠87)", is_foreign), ("eksik", is_missing)]:
    print(f"  {name}: {mask.sum():,} işlem, fraud oranı %{geo_check.loc[mask, 'isFraud'].mean()*100:.3f}")
print()

print("dist1 kovalara göre fraud oranı (zayıf, monotonik değil, kullanılmayacak):")
bins = [-1, 0, 3, 8, 24, 100, 300, 1000, 100000]
labels = ["0", "1-3", "4-8", "9-24", "25-100", "101-300", "301-1000", "1000+"]
geo_check["dist1_bucket"] = pd.cut(geo_check["dist1"], bins=bins, labels=labels)
print((geo_check.groupby("dist1_bucket", observed=True)["isFraud"].mean() * 100).round(3))
print(f"  dist1 eksik: %{geo_check.loc[geo_check['dist1'].isna(), 'isFraud'].mean()*100:.3f}")
print()

p_tld = geo_check["P_emaildomain"].str.split(".").str[-1]
foreign_tld = ~p_tld.isin(["com", "net", "org", "edu"]) & p_tld.notna()
print("P_emaildomain ülke-kodu benzeri TLD (işe yaramıyor, kullanılmayacak):")
print(f"  yabancı-görünümlü TLD: %{geo_check.loc[foreign_tld, 'isFraud'].mean()*100:.3f}")
print(f"  com/net/org/edu: %{geo_check.loc[p_tld.isin(['com','net','org','edu']), 'isFraud'].mean()*100:.3f}")

addr2, yerli/yabancı/eksik dağılımı ve fraud oranı:


  yerli (87): 520,481 işlem, fraud oranı %2.397
  yabancı (≠87): 4,353 işlem, fraud oranı %10.223
  eksik: 65,706 işlem, fraud oranı %11.781

dist1 kovalara göre fraud oranı (zayıf, monotonik değil, kullanılmayacak):
dist1_bucket
0           1.917
1-3         1.790
4-8         1.693
9-24        1.705
25-100      2.039
101-300     3.855
301-1000    3.476
1000+       2.469
Name: isFraud, dtype: float64
  dist1 eksik: %4.516



P_emaildomain ülke-kodu benzeri TLD (işe yaramıyor, kullanılmayacak):
  yabancı-görünümlü TLD: %3.021
  com/net/org/edu: %3.609


**`addr2`, 4,26x (yabancı) ve 4,91x (eksik) oranlarıyla projedeki en güçlü sinyal, ve ilk kez,
naif beklentiyle (yabancı=riskli) aynı yönde.** `dist1` zayıf ve monotonik değil. `P_emaildomain`
TLD'si beklenenin tam tersi yönde (yabancı-TLD %3,02 < yerli-TLD %3,61); kullanılmayacak. Eksik
`addr2`'nin (%11,13 hacim) yabancıdan bile yüksek fraud oranı taşıması (%11,78 vs %10,22), bu
maddeyi ikili değil üç katmanlı (yerli/yabancı/eksik) kurmamızın sebebi.

### Yöntem 1: Sabit politika çarpanı (etiketsiz, sadece bilinen yabancı)

In [27]:
from src.services.anomaly.geographic_context import apply_geographic_policy_adjustment

geographic_policy_adjusted = apply_geographic_policy_adjustment(final_raw, raw_geo)

print("Çarpanın aldığı benzersiz değerler:", sorted(geographic_policy_adjusted["geographic_policy_multiplier"].unique()))
print(geographic_policy_adjusted["geography"].value_counts())
print()
print("Eksik satırlarda çarpan hep 1,0 mı (tasarım gereği, dokunulmuyor):",
      (geographic_policy_adjusted.loc[geographic_policy_adjusted.geography == "missing", "geographic_policy_multiplier"] == 1.0).all())

Çarpanın aldığı benzersiz değerler: [np.float64(1.0), np.float64(2.0)]
geography
domestic    520481
missing      65706
foreign       4353
Name: count, dtype: int64

Eksik satırlarda çarpan hep 1,0 mı (tasarım gereği, dokunulmuyor): True


Çarpan 2,0: `business_context.py`'nin `MIN_CONFIDENCE_MULTIPLIER` (0,5) sabitinin tersi, yeni bir
sayı icat etmek yerine. Sadece KESİN bilinen yabancı (4.353 işlem) etkileniyor; eksik `addr2`
(65.706 işlem) bilinçli olarak dokunulmuyor; etiketsiz bir politika "bilinmiyor"u "riskli" diye
varsayamaz. Bu, Yöntem 2'nin ortaya çıkaracağı gerçek örüntüyle (eksik en yüksek risk) kasıtlı bir
gerilim yaratıyor.

### Yöntem 2: Fraud-oranı-kalibreli, üç katman (`isFraud` kullanan, bilinçli istisna)

In [28]:
from src.services.anomaly.geographic_context import apply_geographic_fraud_calibrated_adjustment

geographic_fraud_adjusted = apply_geographic_fraud_calibrated_adjustment(final_raw, raw_geo, isfraud)

print("Çarpanın aldığı benzersiz değerler:", sorted(geographic_fraud_adjusted["geographic_fraud_calibration_multiplier"].unique()))

Çarpanın aldığı benzersiz değerler: [np.float64(1.0), np.float64(4.264479677294943), np.float64(4.914583784457134)]


Üç çarpan: yerli=1,0, yabancı≈4,26, eksik≈4,91; Yöntem 1'in gözden kaçırdığı tam da eksik verinin
gerçekte yabancıdan bile riskli olduğunu yansıtıyor.

### Doğrulama: üst-%1 ve üst-%5'te ne değişiyor

In [29]:
geo_check_scores = final_raw.merge(
    geographic_policy_adjusted[["TransactionID", "geographic_adjusted_score_policy"]], on="TransactionID"
).merge(
    geographic_fraud_adjusted[["TransactionID", "geographic_adjusted_score_fraud_calibrated"]], on="TransactionID"
)

for pct, label in [(0.01, "üst-%1"), (0.05, "üst-%5")]:
    raw_top = set(geo_check_scores.loc[geo_check_scores["final_raw_anomaly_score"] >= geo_check_scores["final_raw_anomaly_score"].quantile(1 - pct), "TransactionID"])
    print(f"--- {label} (küme boyutu: {len(raw_top):,}) ---")
    for c in ["geographic_adjusted_score_policy", "geographic_adjusted_score_fraud_calibrated"]:
        t = set(geo_check_scores.loc[geo_check_scores[c] >= geo_check_scores[c].quantile(1 - pct), "TransactionID"])
        print(f"  {c}: {len(raw_top & t):,} ortak ({len(raw_top & t) / len(raw_top):.1%})")
    print()

--- üst-%1 (küme boyutu: 5,906) ---
  geographic_adjusted_score_policy: 2,790 ortak (47.2%)


  geographic_adjusted_score_fraud_calibrated: 50 ortak (0.8%)

--- üst-%5 (küme boyutu: 29,527) ---
  geographic_adjusted_score_policy: 26,200 ortak (88.7%)
  geographic_adjusted_score_fraud_calibrated: 2,448 ortak (8.3%)



**Fraud-kalibreli yöntem üst-%1'i neredeyse tamamen değiştiriyor (%0,8 korunuyor):** bu projedeki
en büyük yeniden-sıralama etkisi, beklenen: 70.059 işlem (yabancı+eksik, verinin %11,9'u) 4,26-4,91x
artırılıyor, bu da üst kümenin bileşimini kökten değiştiriyor. Politika yöntemi çok daha ölçülü
(%47,2 / %88,7 korunuyor); sadece 4.353 işlemi (verinin %0,74'ü) etkiliyor.

### Betimleyici kontrol: fraud oranıyla hizalanma, kovalara göre

In [30]:
geo_check_fraud = geo_check_scores.merge(isfraud, on="TransactionID")

for col in ["final_raw_anomaly_score", "geographic_adjusted_score_policy", "geographic_adjusted_score_fraud_calibrated"]:
    geo_check_fraud["decile"] = pd.qcut(geo_check_fraud[col], 10, labels=False, duplicates="drop") + 1
    rates = geo_check_fraud.groupby("decile")["isFraud"].mean() * 100
    print(col)
    print({k: round(v, 2) for k, v in rates.items()})
    print()

final_raw_anomaly_score
{1: 1.82, 2: 2.01, 3: 2.16, 4: 2.69, 5: 3.08, 6: 3.51, 7: 4.29, 8: 4.51, 9: 5.57, 10: 5.34}

geographic_adjusted_score_policy
{1: 1.83, 2: 1.99, 3: 2.17, 4: 2.64, 5: 3.05, 6: 3.46, 7: 4.17, 8: 4.51, 9: 5.58, 10: 5.58}



geographic_adjusted_score_fraud_calibrated
{1: 1.75, 2: 1.68, 3: 1.72, 4: 2.06, 5: 2.16, 6: 2.48, 7: 2.76, 8: 3.13, 9: 4.78, 10: 12.46}



**En keskin üst-kova ayrımı projede şimdiye kadar burada:** fraud-kalibreli coğrafi skorun 10.
kovası %12,46; herhangi bir önceki skor varyantından (en yakın rakip Trust'ın %6,02'si) çok daha
yüksek. Beklenen: bu, projedeki en güçlü ölçülen sinyal (4,26-4,91x), ve etikete göre kalibre edildi
yine de büyüklüğü not edilmeye değer. Politika yöntemi de (etiketsiz, sadece 2,0x sabit çarpan)
ham skordaki hafif 9→10 dalgalanmasını düzeltiyor (%5,57→%5,58, artık monotonik); küçük ama
`isFraud` kullanmadan elde edilen gerçek bir iyileşme.



## 6. False Positive Tablosu: Coğrafi Risk Dahil (Güncellenmiş)

Madde 4'teki sabit-bütçe tablosu, coğrafi risk henüz kurulmadan önce hesaplanmıştı. Aynı yöntemi
(aynı `fp_reduction_table` fonksiyonu) 10 skor varyantının tamamına (8 eski + 2 yeni coğrafi)
yeniden uyguluyoruz.

In [31]:
fp_base_full = fp_base.merge(
    geographic_policy_adjusted[["TransactionID", "geographic_adjusted_score_policy"]], on="TransactionID"
).merge(
    geographic_fraud_adjusted[["TransactionID", "geographic_adjusted_score_fraud_calibrated"]], on="TransactionID"
)

score_variants_full = score_variants + ["geographic_adjusted_score_policy", "geographic_adjusted_score_fraud_calibrated"]


def fp_reduction_table_full(budget_pct):
    budget = round(len(fp_base_full) * budget_pct)
    rows = []
    raw_fp = None
    for col in score_variants_full:
        flagged = fp_base_full.nlargest(budget, col)
        tp = int(flagged["isFraud"].sum())
        fp = budget - tp
        if col == "final_raw_anomaly_score":
            raw_fp = fp
        rows.append({
            "score": col, "TP": tp, "FP": fp,
            "precision_%": round(tp / budget * 100, 2),
            "recall_%": round(tp / total_fraud * 100, 2),
            "FP_degisim": fp - raw_fp,
        })
    return pd.DataFrame(rows).set_index("score")


print(f"Alert bütçesi: üst-%5 ({round(len(fp_base_full)*0.05):,} işlem)")
print(fp_reduction_table_full(0.05))

Alert bütçesi: üst-%5 (29,527 işlem)


                                                TP     FP  precision_%  \
score                                                                    
final_raw_anomaly_score                       1481  28046         5.02   
business_adjusted_score_confidence_based      1369  28158         4.64   
business_adjusted_score_fraud_calibrated      1575  27952         5.33   
weekend_adjusted_score_confidence_based       1497  28030         5.07   
weekend_adjusted_score_fraud_calibrated       1465  28062         4.96   
weekend_peak_adjusted_score_confidence_based  1454  28073         4.92   
weekend_peak_adjusted_score_fraud_calibrated  1761  27766         5.96   
trust_adjusted_score_confidence_based         1278  28249         4.33   
trust_adjusted_score_fraud_calibrated         1683  27844         5.70   
geographic_adjusted_score_policy              1662  27865         5.63   
geographic_adjusted_score_fraud_calibrated    4073  25454        13.79   

                                     

**Coğrafi fraud-kalibreli skor, tüm 10 varyant arasında en güçlü FP azaltımını veriyor:**
beklenen, çünkü şimdiye kadarki en keskin ölçülen sinyal (4,26-4,91x) burada. Coğrafi politika
çarpanı (etiketsiz, sadece 2,0x, sadece %0,74'lük dilime) daha ölçülü ama yine de gerçek bir
iyileşme; Madde 4'teki 4 etiketsiz kuraldan sadece 1'inin (hafta sonu) iyileştirdiği tabloya
karşı, bu ikinci etiketsiz kural da iyileştiren tarafta yer alıyor.

In [32]:
print(f"Alert bütçesi: üst-%1 ({round(len(fp_base_full)*0.01):,} işlem)")
print(fp_reduction_table_full(0.01))

Alert bütçesi: üst-%1 (5,905 işlem)


                                               TP    FP  precision_%  \
score                                                                  
final_raw_anomaly_score                       166  5739         2.81   
business_adjusted_score_confidence_based      166  5739         2.81   
business_adjusted_score_fraud_calibrated      226  5679         3.83   
weekend_adjusted_score_confidence_based       166  5739         2.81   
weekend_adjusted_score_fraud_calibrated       241  5664         4.08   
weekend_peak_adjusted_score_confidence_based  166  5739         2.81   
weekend_peak_adjusted_score_fraud_calibrated  482  5423         8.16   
trust_adjusted_score_confidence_based         166  5739         2.81   
trust_adjusted_score_fraud_calibrated         229  5676         3.88   
geographic_adjusted_score_policy              443  5462         7.50   
geographic_adjusted_score_fraud_calibrated    671  5234        11.36   

                                              recall_%  FP_degi

Üst-%1'de coğrafi fraud-kalibreli skor da en güçlü azaltımı veriyor; tutarlı.

## 7. ROC-AUC: dört düzeltmenin de tek bir tabloda karşılaştırılması

Yukarıdaki her düzeltme kendi bölümünde üst-%1/%5 sabit-bütçeli false-positive karşılaştırmasıyla
değerlendirildi; bu, sadece iki eşiğe bakıyor. ROC-AUC tüm eşik aralığını tek bir sayıya
sıkıştırıyor: bir düzeltmenin üst-%1'de iyi görünüp üst-%10'da kötüleşip kötüleşmediğini de
yakalar. `fp_base_full` zaten tüm 10 skor varyantını ve `isFraud`'u aynı tabloda taşıyor;
`compare_auc` doğrudan uygulanabilir.

In [33]:
from src.services.evaluation.roc import compare_auc

compare_auc(fp_base_full, score_variants_full)

,score_column,auc
0,geographic_adjusted_score_fraud_calibrated,0.691500
1,trust_adjusted_score_fraud_calibrated,0.617648
2,business_adjusted_score_fraud_calibrated,0.614994
3,geographic_adjusted_score_policy,0.613212
4,weekend_peak_adjusted_score_fraud_calibrated,0.612789
5,weekend_adjusted_score_fraud_calibrated,0.609941
6,final_raw_anomaly_score,0.609618
7,weekend_adjusted_score_confidence_based,0.598974
8,weekend_peak_adjusted_score_confidence_based,0.596719
9,business_adjusted_score_confidence_based,0.580522


**Genel olarak AUC, fixed-bütçe bulgularını doğruluyor; ama bir yerde gerçek bir nüans ortaya
çıkarıyor, gizlenmiyor.**

**Tutarlı olanlar:** geographic fraud-calibrated (0,6915) açık ara en güçlü, ham skoru (0,6096)
çok geride bırakıyor; geographic policy (etiketsiz, 0,6132) ham skorun üzerinde, tek etiketsiz
düzeltme ki bunu tüm eşik aralığında da başarıyor. `business_adjusted_score_confidence_based`
(0,5805) ve `trust_adjusted_score_confidence_based` (0,5606) ikisi de ham skorun altında; Madde 4'ün
"bu ikisi false positive'i artırıyor" bulgusuyla tam örtüşüyor.

**Nüans burada: hafta sonu düzeltmesi.** Madde 4, `weekend_adjusted_score_confidence_based`'i
üst-%5 sabit bütçesinde FP'yi azaltan (-16) tek etiketsiz kural olarak raporladı. Ama aynı skorun
AUC'u (0,5990) ham skorun (0,6096) ALTINDA; `weekend_peak_adjusted_score_confidence_based` da
aynı şekilde (0,5967). Yani bu düzeltme üst-%5'lik dilimde gerçekten iyileşiyor, ama tüm eşik
aralığında (özellikle üst-%5'in dışında) genel sıralama kalitesi hafifçe kötüleşiyor; iki metrik
burada anlaşmıyor. Bu, "sabit bütçe" ve "tüm eşik aralığı" sorularının GERÇEKTEN farklı sorular
olduğunun somut kanıtı: bir düzeltme birinde iyi görünüp diğerinde kötü çıkabiliyor, ve bu proje
ikisini de raporluyor, birini diğerine feda etmeden.

---

**Durum:** Case 6 tamamlandı. Brief'te listelenen dört context boyutu da (business hours, weekend,
trusted entity, geographic risk) kuruldu, her biri etiketsiz ve fraud-kalibreli iki yöntemle
karşılaştırıldı, ve hepsinin gerçek false-positive etkisi ortak bir sabit-bütçe çerçevesinde
ölçüldü. En önemli genel bulgu: "mantıklı görünen" bir iş kuralı garanti değil. Dört etiketsiz
kuraldan ikisi (hafta sonu, coğrafi politika) üst-%5 sabit bütçesinde gerçekten iyileştirdi,
ikisi (business hours, trusted entity) kötüleştirdi; en güçlü tek sinyal coğrafi risk oldu, hem
etiketsiz hem etiket-kalibreli versiyonunda. Bölüm 7'nin ROC-AUC karşılaştırması bunu büyük ölçüde
doğruladı, ama bir nüans ekledi: hafta sonu düzeltmesi sabit bütçede iyileşse de tüm eşik
aralığında (AUC) ham skorun hafifçe altında kalıyor; "sabit bütçe" ve "tüm eşik aralığı" farklı
sorular, ikisi de raporlandı. Tek bir "üretime taşınacak" birleşik skor burada oluşturulmadı;
bu, ölçümlerin ışığında ayrıca karar verilecek bir konu olarak bırakıldı.